In [65]:
import pandas as pd
print("Environment is ready!")

Environment is ready!


Web Scraping
Use google-play-scraper to collect reviews, ratings, dates, and app names for all three banks.
Target a minimum of 400+ reviews per bank (1,200 total). If the library returns fewer, expand the date range or document the limitation clearly.
Collect the following fields: review text, rating (1–5), review date, bank / app name, source ("Google Play").


**# # Web Scraping**

In [66]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [67]:
# The unique identifier for CBE's app on the Google Play Store
CBE_ID = 'com.combanketh.mobilebanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    CBE_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("CBE App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

CBE App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.284369
Total Ratings: 48,291
Total Reviews: 9,305
Installs     : 5,000,000+


In [68]:
# Step 2: Scrape reviews
print(f"Scraping reviews for CBE...")

result, continuation_token = reviews(
    CBE_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=450,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for CBE...
Collected 450 raw reviews


In [74]:
# Step 3: Extract required fields from reviews
cbe_reviews_data = []

for review in result:
    cbe_reviews_data.append({
        'review_id': review.get('reviewId', ''),        # Review ID for deduplication
        'review_text': review.get('reviewText', review.get('content', '')),  # Review content
        'rating': review.get('score', 0),                 # Rating (1-5)
        'review_date': review.get('at', ''),              # Review date
        'bank': 'CBE Bank',                               # Bank/app name
        'app_name': 'CBE BANK',                            # App name in capital letters
        'source': 'Google Play'                           # Source
    })

cbe_reviews = pd.DataFrame(cbe_reviews_data)
print(f"\nRaw reviews before deduplication: {len(cbe_reviews)}")
cbe_reviews = cbe_reviews.drop_duplicates(subset=['review_id'], keep='first').reset_index(drop=True)
print(f"Reviews after deduplication by review_id: {len(cbe_reviews)}")

print("\n" + "=" * 50)
print("CBE Reviews DataFrame")
print("=" * 50)
print(f"Total reviews collected: {len(cbe_reviews)}")
print(f"\nColumns: {list(cbe_reviews.columns)}")
print(f"\nFirst 3 reviews:")
print(cbe_reviews[['bank', 'rating', 'review_date', 'source']].head(3))
print(f"\nRating distribution:")
print(cbe_reviews['rating'].value_counts().sort_index())



Raw reviews before deduplication: 450
Reviews after deduplication by review_id: 450

CBE Reviews DataFrame
Total reviews collected: 450

Columns: ['review_id', 'review_text', 'rating', 'review_date', 'bank', 'app_name', 'source']

First 3 reviews:
       bank  rating         review_date       source
0  CBE Bank       5 2026-05-13 06:18:45  Google Play
1  CBE Bank       5 2026-05-13 02:19:17  Google Play
2  CBE Bank       2 2026-05-13 01:27:52  Google Play

Rating distribution:
rating
1     65
2     10
3     31
4     37
5    307
Name: count, dtype: int64


In [75]:
# Step 4: Prepare final dataset (CBE Bank only)

print(f"\n{'=' * 50}")
print("FINAL DATASET - CBE BANK")
print(f"{'=' * 50}")
print(f"Total reviews: {len(cbe_reviews)}")
print(f"\nColumns: {list(cbe_reviews.columns)}")
print(f"\nDataset info:")
print(cbe_reviews.info())
print(f"\nRating distribution:")
print(cbe_reviews['rating'].value_counts().sort_index())



FINAL DATASET - CBE BANK
Total reviews: 450

Columns: ['review_id', 'review_text', 'rating', 'review_date', 'bank', 'app_name', 'source']

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   review_id    450 non-null    str           
 1   review_text  450 non-null    str           
 2   rating       450 non-null    int64         
 3   review_date  450 non-null    datetime64[us]
 4   bank         450 non-null    str           
 5   app_name     450 non-null    str           
 6   source       450 non-null    str           
dtypes: datetime64[us](1), int64(1), str(5)
memory usage: 24.7 KB
None

Rating distribution:
rating
1     65
2     10
3     31
4     37
5    307
Name: count, dtype: int64


In [76]:
# View first 10 rows of the required fields only
required_columns = ['review_text', 'rating', 'review_date', 'bank', 'app_name', 'source', 'review_id']

# Reorder the DataFrame to ensure the columns appear in the requested order
cbe_reviews = cbe_reviews[required_columns]

print("Required columns:")
print(cbe_reviews.columns.tolist())
print("\nFirst 10 reviews with the required fields:")
print(cbe_reviews.head(10))

print("\nReview text availability:")
print(f"Non-empty review_text rows: {cbe_reviews['review_text'].astype(bool).sum()}")
print(f"Empty review_text rows: {len(cbe_reviews) - cbe_reviews['review_text'].astype(bool).sum()}")

if len(cbe_reviews) > 0:
    print("\nFirst raw review object keys:")
    print(result[0].keys())
    print("\nFirst raw review object sample:")
    print(result[0])


Required columns:
['review_text', 'rating', 'review_date', 'bank', 'app_name', 'source', 'review_id']

First 10 reviews with the required fields:
                                         review_text  rating  \
0                                            is good       5   
1                                                wow       5   
2                                   Good application       2   
3  Nice, but I can't get some recently transactio...       5   
4  Very Secure but very poor interface and limite...       1   
5                                     very nice 100%       5   
6                                               good       5   
7                                        Good to use       5   
8                                                cbe       1   
9                                                Cbe       4   

          review_date      bank  app_name       source  \
0 2026-05-13 06:18:45  CBE Bank  CBE BANK  Google Play   
1 2026-05-13 02:19:17  CBE Bank  

# # Preprocessing

In [77]:
#lets see the first few rows of the final dataset
cbe_reviews.head()

,review_text,rating,review_date,bank,app_name,source,review_id
0,is good,5,2026-05-13 06:18:45,CBE Bank,CBE BANK,Google Play,ff53332b-2e76-46d6-83d3-f93f968a4b18
1,wow,5,2026-05-13 02:19:17,CBE Bank,CBE BANK,Google Play,363a5616-ed3d-4274-85ee-77071067f81d
2,Good application,2,2026-05-13 01:27:52,CBE Bank,CBE BANK,Google Play,56185597-d29b-4a60-a0fb-6783638230a7
3,"Nice, but I can't get some recently transactio...",5,2026-05-13 01:21:48,CBE Bank,CBE BANK,Google Play,35efe702-40c9-4e46-ad67-7574ab9ef42d
4,Very Secure but very poor interface and limite...,1,2026-05-13 01:11:25,CBE Bank,CBE BANK,Google Play,b41cb49d-59d3-41b8-bbb2-b1952005951b


In [78]:
# Remove duplicate reviews by review_id
cbe_reviews = cbe_reviews.drop_duplicates(subset=['review_id'], keep='first').reset_index(drop=True)

print(f"Total reviews after removing duplicates: {len(cbe_reviews)}")
print(f"Final dataset shape: {cbe_reviews.shape}")

Total reviews after removing duplicates: 450
Final dataset shape: (450, 7)


Handle missing values

In [79]:
#drop rows missing review text or rating; document counts.
cbe_reviews = cbe_reviews.dropna(subset=['review_text', 'rating'])
print(f"Total reviews after dropping missing values: {len(cbe_reviews)}")
print(f"Final dataset shape: {cbe_reviews.shape}")

Total reviews after dropping missing values: 450
Final dataset shape: (450, 7)


In [ ]:
#Normalize dates to YYYY-MM-DD format.
cbe_reviews['review_date'] = pd.to_datetime(cbe_reviews['review_date'], errors='coerce').dt.strftime('%Y-%m-%d')
print(f"Final dataset shape: {cbe_reviews.shape}")


Final dataset shape: (450, 7)


In [85]:
# 2. Get the range
start_date = cbe_reviews['review_date'].min()
end_date = cbe_reviews['review_date'].max()

print(f"Data spans from: {start_date}")
print(f"Up to: {end_date}")


Data spans from: 2026-03-09
Up to: 2026-05-13


In [81]:
#save the cleaned dataset as a CSV with columns: review, rating, date, bank, source.
cbe_reviews.to_csv('cbe_reviews_cleaned.csv', index=False)  